Auditoria Gran Encuesta Integrada De Hogares 2026

4 MODULOS:
-CGSE
-FT
-OCUPADOS
-DESEMPLEADOS

In [0]:
#Leyendo la tabla de caracteristicas generales
cgse = spark.read.format("delta").table("workspace.electiva.cgse")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
ocupados = spark.read.format("delta").table("workspace.electiva.ocupados")

In [0]:
cgse.display()

In [0]:
# Convertir el DataFrame de Spark a Pandas
cgse = cgse.toPandas()
desempleados=desempleados.toPandas()
ocupados=ocupados.toPandas()

In [0]:
cgse

In [0]:
cgse['FEX_C18'].sum()

In [0]:
#Leyendo la tabla de fuerza de trabajo
ft = spark.read.format("delta").table("workspace.electiva.fuerzadetrabajo")
ft=ft.toPandas()
ft

* DIRECTORIO : El directorio identifica la vivienda.
* SECUENCIA_P : La secuencia_p identifica el hogar.
* ORDEN : La variable orden hace referencia a las personas.

In [0]:
# Filtrando a nivel de columna
cgse['PERIODO']

#Aplicando operaciones
cgse['PERIODO'].sum()
cgse['PERIODO'].max()
cgse['PERIODO'].min()
cgse['PERIODO'].mean()
cgse['PERIODO'].std()
cgse['PERIODO'].var()
cgse['PERIODO'].median()
cgse['PERIODO'].mode()

In [0]:
#Primeros n registros de la tabla
cgse.head(20)

In [0]:
cgse[['PERIODO','FEX_C18','P6090']]

In [0]:
#Filtro a nivel de fila, filtrar solo personas donde la variable DPTO=63
cgse[cgse['DPTO']==63]
cgse_quindio=cgse[cgse['DPTO']==63]

#Operadores de comparacion
"""
== : igual a
< :  menor a 
<= : menor o igual a 
 : mayor a 
= : mayor o igual a 
!= : diferente a

"""

In [0]:
cgse['DPTO'].unique()

In [0]:
cgse['P3271'].unique()

**1. Población total**

In [0]:
cgse
PT=cgse['FEX_C18'].sum()
PT

Primer tema a resolver como calcular la cnatidad de personas desempleadas

In [0]:
desempleados

In [0]:
D = desempleados[desempleados['DSI']==1]['FEX_C18'].sum()
D

segundo tema a resolver, como calcular la PEA?

In [0]:
PEA = ft[ft['FT']==1]['FEX_C18'].sum()
PEA

In [0]:
TD = D/PEA*100
print(f'La tasa de desempleo en colombia para junio 2026 es de {TD} % ')

ACTIVIDAD La tasa global de participación se ubicó en 64,5%
la tasa de ocupación fue 59,4%
calcular Tasa global de partipacion -> TGP = PEA/PET*100
calcular tasa de ocupacion -> TO = O/PEA100,TO = O/PET100

In [0]:
PET = ft[ft['PET']==1]['FEX_C18'].sum()
PET

In [0]:
#tasa de participación global
TGP = PEA/PET*100
print(f'La tasa de PARTICIPACIÓN GLOGAL en colombia para junio 2026 es de {TGP} % ')

In [0]:
O = ocupados[ocupados['OCI']==1]['FEX_C18']. sum()

In [0]:
#tasa de ocupados con PEA y PET
TO = O/PEA*100
TO = O/PET*100
print(f'La tasa de ocupación en colombia para junio 2026 es de {TO} %')

In [0]:
ocupados['RAMA2D_R4'].unique()

In [0]:
#Calculando cuantas personas trabajan en el sector de alojamiento y servicios de comida en colombia a junio de 2026
ocupados[(ocupados['RAMA2D_R4']==55) | (ocupados['RAMA2D_R4']==56) ]['FEX_C18'].sum()
# En colombia en el sector de alojamiento y servicios de comida trabajan 2.057.354 personas a junio de 2026


In [0]:
#Calculando cuantas personas trabajan en el sector contruccion y trabajo en colombia a junio de 2026
ocupados[(ocupados['RAMA2D_R4']==41) | (ocupados['RAMA2D_R4']==42) | (ocupados['RAMA2D_R4']==43)]['FEX_C18'].sum()
# En colombia en el sector construcción trabajo 1.678.387 personas a junio de 2026

In [0]:
#Calculando cuantas personas trabajan en el sector de agricultura , ganaderia, caza, silvicultura y pesca en colombia a junio de 2026
ocupados[(ocupados['RAMA2D_R4']==1) | (ocupados['RAMA2D_R4']==2) | (ocupados['RAMA2D_R4']==3)]['FEX_C18'].sum()
# En colombia en el sector agricultura , ganaderia, caza, silvicultura y pesca hay 3'280.869 personas a junio de 2026

In [0]:
desempleados

In [0]:
cgse

In [0]:
cgse[(cgse['DIRECTORIO']==8637743) & (cgse['SECUENCIA_P']==1) & (cgse['ORDEN']==1) ][['DIRECTORIO','SECUENCIA_P','ORDEN','P3271']]

Necesitamos traernos la variable P3271 del modulo cgse y añadirla al modulo de desempleados para doder calcular tasa de desempleo en hombres y mujeres del quindio

In [0]:
import pandas as pd#Primero hay que anexar la P3271 a la tabla de desempleados

desemplados = pd.merge(desempleados, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271', 'P6040']],
                       on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')

In [0]:
desemplados

In [0]:
#cantidad de mujeres desempleadas en el quindio
MDQ = desemplados[(desemplados['P3271']==2) & (desemplados['DSI']==1) & (desempleados['DPTO']==63)]['FEX_C18'].sum()
# y de hombres
HDQ = desemplados[(desemplados['P3271']==1) & (desemplados['DSI']==1) & (desempleados['DPTO']==63)]['FEX_C18'].sum()
print (f" en el quindio hay {MDQ} Mujeres desempleadas")
print (f" en el quindio hay {HDQ} Hombres desempleados")


In [0]:
ft = pd.merge(ft, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271', 'P6040']],
                       on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')

In [0]:
#Calcular PEA de mujeres y hombres en el Quindio
PEAMQ = ft[(ft['FT']==1) & (ft[ 'DPTO']==63) & (ft['P3271']==2)]['FEX_C18'].sum()
#PEA de hombres en el Quindio
PEAHQ = ft[(ft['FT']==1) & (ft[ 'DPTO']==63) & (ft['P3271']==1) ]['FEX_C18'].sum()
print (f"en el Quindio hay {PEAMQ} mujeres en la PEA")
print (f"en el Quindio hay {PEAHQ} hombres en la PEA")

In [0]:
# Por ultimo cuando ya tenemos cantidad de desempleados y PEA podemos calcular el porcentaje de desempleo
TOHQ = HDQ/PEAHQ *100
print(f"El porcentaje de desempleo masculino en el quindio es de {TOHQ}%")
TDMQ = MDQ/PEAMQ *100
print(f"El porcentaje de desempleo femenino en el quindio es de {TDMQ}%")